In [ ]:
# 第1步：选择本次实验，并在首次运行时读取已经推送的最新 Git 代码。
# 代码放在 /tmp，避免整个仓库被 Kaggle 当作实验结果导出；本次运行中不更新代码。
# 提交账号使用 daisychen2，挂载该账号自有的私有预处理数据包。
import os, sys, json, subprocess, time
from pathlib import Path
QUEUE_ID = "LFP_R1_full"  # 选择任务队列配置中的一个名称；当前为 LFP 完整 RCR，并附带去校准评估。
REPO = Path("/tmp/EnhancedMTADGAT")
if not (REPO / ".git").exists():
    subprocess.run(["git", "clone", "--depth", "1", "--filter=blob:none", "--sparse", "https://github.com/wonkawonka/EnhancedMTADGAT.git", str(REPO)], check=True)
os.chdir(REPO)
# 仅拉取训练源码、配置和固定参考实现，减少无关报告、图片的下载。
subprocess.run(["git", "sparse-checkout", "set", "src", "configs", "external_baselines/reference_snapshots"], check=True)
print("git:", subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip(), flush=True)
# 第2步：使用 Kaggle 自带的 PyTorch 和数值环境，只安装当前缺少的小型依赖。
# 不在这里重装完整环境；实际软件版本会随实验效率记录保存。
import importlib.util
missing = [name for name in ("einops", "psutil", "joblib", "threadpoolctl") if importlib.util.find_spec(name) is None]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", *missing], check=True)
import torch
# 第3步：在已挂载数据集中查找冻结预处理包的顶层清单。
# 这里只定位文件，不重新解析原始 CSV/pickle，也不重新划分数据。
candidates = [p.parent for p in Path("/kaggle/input").glob("*/bundle_manifest.json")]
candidates += [p.parent for p in Path("/kaggle/input").glob("*/*/bundle_manifest.json")]
BUNDLE = next((p for p in candidates if set(json.loads((p/"bundle_manifest.json").read_text()).get("datasets", {})) == {"LFP", "NCM", "Brand1"}), None)
assert BUNDLE is not None, "Attach the private battery-formal-prepared-20261113 dataset"
# 第4步：为扁平上传文件建立目录软链接，不改变任何数据内容或划分。
if not (BUNDLE / "LFP").is_dir():
    view = Path("/tmp/battery-prepared-view")
    for ds in ("LFP", "NCM", "Brand1"):
        (view / ds).mkdir(parents=True, exist_ok=True)
        for file in BUNDLE.glob(ds + "__*"):
            target = view / ds / file.name[len(ds)+2:]
            if not target.exists():
                target.symlink_to(file)
    BUNDLE = view
# 第5步：读取已确认的正式协议与任务清单；窗长、步长、种子和轮数由配置决定。
# 训练入口还会核验数据包哈希、样本边界、归一化来源和实际参数。
PLAN = Path("configs/external/battery_kaggle_formal_20261112_v1.json")
queue = json.loads(Path("configs/external/battery_kaggle_queue_20261113.json").read_text())["queues"][QUEUE_ID]
# 第6步：按模型选择设备。Isolation Forest 使用 CPU，其余正式模型使用 GPU。
# 神经网络未分配 GPU 时在这里停止，避免误用 CPU 运行数小时。
CPU_ONLY = all(j["experiment"] == "IsolationForest" for j in queue)
DEVICE = "cpu" if CPU_ONLY else "cuda:0"
if not CPU_ONLY:
    assert torch.cuda.is_available(), "Enable GPU before formal neural experiments"
    print("GPU:", torch.cuda.get_device_name(0), "total GiB:", torch.cuda.get_device_properties(0).total_memory / 2**30, flush=True)
else:
    print("Isolation Forest uses CPU; GPU is not required", flush=True)
# 第7步：建立统一结果目录。模型、指标、效率记录和压缩包都写入 artifacts。
# 预处理输入仍从 /kaggle/input 只读加载，不复制到导出目录。
OUT = Path("/kaggle/working/artifacts")
OUT.mkdir(parents=True, exist_ok=True)
print("queue:", QUEUE_ID, queue, "bundle:", BUNDLE, flush=True)


In [ ]:
from src.experiments.battery_data import write_json
from src.experiments.artifacts import summarize_output
# 第8步：准备执行与恢复路径。空输出目录表示从头训练。
# 如需断点恢复，须先把同一协议、代码版本的完整恢复产物放回原输出目录。
# 只挂载旧数据集或只提供模型权重，不等于已经恢复优化器、随机状态和训练进度。
ds = queue[0]["dataset"]
assert all(j["dataset"] == ds for j in queue)
evidence = OUT / "checks" / QUEUE_ID
cmd = [sys.executable, "-u", "-m", "src.runners.run_battery_formal", "--device", DEVICE]
start = time.monotonic()
# 第9步：先用合成小样本核验本队列的原生损失、梯度、评分和归档路径。
# 使用声明的实际 batch 检查显存/API，检查用权重会丢弃，不作为正式训练结果。
# 这一步通过后，正式入口仍会核验实际挂载的数据包和真实首批输入。
print("Starting bounded native loss/score/gradient smoke", flush=True)
subprocess.run(cmd + ["--plan", str(PLAN), "--smoke", "--dataset", ds,
    "--smoke-experiments", ",".join(j["experiment"] for j in queue), "--output", str(evidence)], check=True)
# 第10步：记录本次队列、预期单元、实际 Git commit、数据路径及完成状态。
# 若 Kaggle 提供脚本版本号则一并记录；下载时还需从平台核对实际版本。
manifest = dict(queue_id=QUEUE_ID, expected_cells=queue,
    git_commit=subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip(),
    bundle=str(BUNDLE), script_version_id=os.environ.get("KAGGLE_SCRIPT_VERSION_ID"),
    protocol="battery_kaggle_formal_20261112_v1", completed_cells=[], status="running")
write_json(OUT / "protocol_manifest.json", manifest)
# 第11步：设置本次神经网络训练循环的软时间预算，给后续评分和打包留出余量。
# 6小时不是总训练轮数上限；到时保存完整恢复状态并暂停，后续可继续。
# 校准、最终评分、压缩不计入该软预算；Isolation Forest 的 fit 不使用此训练循环预算。
# 一个队列包含一个训练/拟合单元；完整 RCR 队列另附带不重训的去校准评估。
TRAINING_ALLOWANCE_SECONDS = 6 * 3600
# 第12步：依次执行本队列的正式单元；保持已确认的训练、评分和阈值规则。
# 子进程实时输出关键进度，结束后汇总当前产物中的指标与效率 CSV。
for job in queue:
    args = cmd + ["--plan", str(PLAN), "--bundle", str(BUNDLE), "--dataset", job["dataset"],
        "--experiment", job["experiment"], "--output", str(OUT),
        "--smoke-evidence", str(evidence / "smoke.json"), "--max-seconds", str(TRAINING_ALLOWANCE_SECONDS)]
    print("Starting:", job, flush=True)
    result = subprocess.run(args)
    summarize_output(OUT)
    # 返回码75表示已保存恢复状态后暂停，不能把它写成实验完成。
    if result.returncode == 75:
        manifest["status"] = "paused_with_recovery"
        write_json(OUT / "protocol_manifest.json", manifest)
        print("Training allowance reached; full recovery archive saved. Do not label complete.", flush=True)
        break
    # 其他非零返回码表示失败：保留证据并停止，不自动缩数据、改 batch 或换公式。
    if result.returncode:
        manifest["status"] = "failed"
        write_json(OUT / "protocol_manifest.json", manifest)
        raise RuntimeError(f"Cell failed, preserved artifacts: {job}")
    # 只有子进程完成评估、核验必要产物并打包成功，才登记该单元完成。
    manifest["completed_cells"].append(job)
    write_json(OUT / "protocol_manifest.json", manifest)
else:
    manifest["status"] = "complete"
# 第13步：保存最终状态和本次总耗时，并列出已生成的压缩包。
# archives 下的小结果包用于看表，权重、原生分数和恢复状态分包保存，按需下载。
manifest["wall_seconds"] = time.monotonic()-start
write_json(OUT / "protocol_manifest.json", manifest)
print("Status:", manifest["status"], "Archives:", [p.name for p in (OUT/"archives").glob("*")], flush=True)
